# Machine Learning #08 — Attention Mechanisms 🎯

**Attention** is a central idea in modern deep learning, including translation systems and
transformer language models. Its core computation fits into **three lines of tensor code**.

Today's workflow:

> **hidden states → scores → softmax weights → weighted sum (context) → prediction**

## Imports and setup

A **GPU** runtime in Colab can speed up training; CPU also works, with hardware-dependent
training time. Use **PyTorch 2.0 or newer** for the scaled dot-product attention examples.
The next cell installs `torchinfo` into the active notebook kernel.

In [ ]:
%pip install -q torchinfo

In [ ]:
import re
import time
import urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import torchinfo

from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110})

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch version:", torch.__version__)
print("device:", device)

# 1. Why attention? 🧠

Think about the everyday meaning of the word **attention**: out of everything around you,
you *select* the few things that matter right now and ignore the rest.
A neural network can do exactly the same with its inputs.

### The problem with recurrent networks 🔎

An LSTM reads a sequence step by step and keeps a **hidden state** $h_t$.
If we only use the **last** hidden state $h_T$ for the prediction, then:

- the whole sequence — no matter how long — must be squeezed into **one fixed-size vector**,
- information from the **beginning** of the sequence has to survive dozens of update steps
  (the classic "forgetting" problem),
- every position contributes to the prediction **implicitly**; we cannot tell which word
  actually mattered.

> 💡 A single vector as the summary of a 200-word review is a real **information bottleneck**.

### The idea 💡

Instead of throwing away $h_1, \dots, h_{T-1}$, we keep **all** hidden states and build the
summary as a **weighted sum**:

$$c \;=\; \sum_{t=1}^{T} \alpha_t \, h_t$$

where the weights $\alpha_t$ are:

- **non-negative** and **sum to one** ($\sum_t \alpha_t = 1$) — because they come from a **softmax**,
- **input-dependent** — learned scoring parameters produce weights for each sequence,
- **inspectable** — a large $\alpha_t$ gives hidden state $h_t$ more weight in the sum. 🔍

The summary is still fixed-size, but it can now draw directly on every hidden state.
An LSTM state already contains information from earlier positions, so a large weight
does **not** prove that the token at that position caused the prediction. Attention
plots are useful diagnostics, not a complete explanation of the model. See
[Jain and Wallace (2019)](https://aclanthology.org/N19-1357/) for this distinction.

### How are the weights computed?

In two steps:

1. **Score** every position with a small learnable function: $e_t = \text{score}(h_t)$
2. **Normalise** the scores into a probability distribution:

$$\alpha_t \;=\; \frac{\exp(e_t)}{\sum_{k=1}^{T}\exp(e_k)} \;=\; \text{softmax}(e)_t$$

That is it. The rest of this notebook is that formula, over and over again. 🙂

> 🔎 **Question:** which real-world tasks would benefit the most from attention?
> Think about translation, medical time series, anomaly detection in sensor data...

# 2. Attention by hand ✋

Before we train anything, let's compute attention **manually** on a tiny example.

Imagine a 4-token sentence that has already been processed by an LSTM with **3 hidden units**.
The result is a matrix of hidden states $H$ with shape `(batch=1, time=4, hidden=3)`.

In [ ]:
tokens = ["the", "food", "was", "terrible"]

# Hidden states H: (batch, time, hidden) - here we simply invent them
H_demo = torch.tensor([[
    [0.1, 0.0, 0.2],   # "the"       - a boring function word
    [0.7, 0.2, 0.1],   # "food"      - topic of the sentence
    [0.0, 0.1, 0.1],   # "was"       - boring again
    [0.9, 0.8, 0.3],   # "terrible"  - carries the sentiment
]])

print("H shape (batch, time, hidden):", H_demo.shape)
H_demo

### Step 1 — score every position

For now we use the simplest possible scoring function: a **dot product with a learnable
vector** $w$ (we just pick the numbers here, in a real model they are trained):

$$e_t = h_t \cdot w$$

In [ ]:
w = torch.tensor([1.0, 2.0, 0.0])          # "importance direction" in hidden space

scores = H_demo @ w                        # (1, 4, 3) @ (3,) -> (1, 4)
print("scores shape:", scores.shape)

for tok, s in zip(tokens, scores[0]):
    print(f"  {tok:>9s}: {s.item():5.2f}")

### Step 2 — softmax turns scores into weights

Softmax makes the scores **positive** and makes them **sum to 1**, so they can be read as
"how much percent of my attention goes here".

In [ ]:
alpha = torch.softmax(scores, dim=1)       # normalise over the TIME axis!

for tok, a in zip(tokens, alpha[0]):
    print(f"  {tok:>9s}: {a.item():.3f}")

print("\nSum of the weights:", alpha.sum().item())   # approximately 1.0, up to floating-point rounding

> ⚠️ Note `dim=1`: we normalise over **time**, not over the hidden units.
> A softmax over the wrong axis is one of the most common attention bugs.

### Step 3 — the context vector

$$c = \sum_t \alpha_t h_t$$

In tensor form this is a **batched matrix multiplication**: `(1, 1, 4) @ (1, 4, 3) -> (1, 1, 3)`.

In [ ]:
context = torch.bmm(alpha.unsqueeze(1), H_demo).squeeze(1)   # (1, 3)

print("context vector      :", context[0].tolist())
print("simple mean pooling :", H_demo.mean(dim=1)[0].tolist())
print("last hidden state   :", H_demo[:, -1, :][0].tolist())

Compare the three rows above:

- **mean pooling** gives each invented hidden state the same coefficient,
- the **last hidden state** selects the final row,
- the **context vector** gives more weight to rows with larger scores under our chosen `w`.

This example illustrates the arithmetic. A real LSTM's last state can also contain
information about earlier words; it is not just a representation of the final word.

In [ ]:
plt.figure(figsize=(5, 2.6))
sns.barplot(x=tokens, y=alpha[0].numpy(), color="tab:blue")
plt.ylabel(r"attention weight $\alpha$")
plt.title("Where does the model look?")
plt.tight_layout()
plt.show()

> 💡 **The whole mechanism is three lines:** `scores -> softmax -> weighted sum`.
> Everything that follows only changes **how the scores are computed** and **who asks
> the question**.

In a real model we do **not** invent the vector `w` — we let the network **learn** it.
That is exactly what we do next.

# 3. Task 1: Find-Max 🔢

A toy task that is perfect for **seeing** attention at work.

- **Input:** a sequence of 15 random numbers from the interval $[0, 10)$
- **Output:** the **maximum** of the sequence (a single number → **regression**)

Why this task? We know the position of the maximum, so we can check whether the
attention peak aligns with it. Alignment is a diagnostic, not a requirement: an LSTM
can also carry the maximum into a later hidden state. 🔍

In [ ]:
def make_sequences(n_samples, seq_len=15, generator=None):
    X = torch.rand(n_samples, seq_len, 1, generator=generator) * 10.0
    y = X.max(dim=1).values
    return X, y

In [ ]:
SEQ_LEN = 15
g = torch.Generator().manual_seed(SEED)

X_train, y_train = make_sequences(6000, SEQ_LEN, g)  
X_val,   y_val   = make_sequences(1000, SEQ_LEN, g)
X_test,  y_test  = make_sequences(1000, SEQ_LEN, g)

print("train:", X_train.shape, y_train.shape)
print("val  :", X_val.shape,   y_val.shape)
print("test :", X_test.shape,  y_test.shape)

In [ ]:
# One example: the sequence and its maximum
print("sequence:", X_train[0].squeeze(-1).numpy().round(2))
print(f"maximum : {y_train[0].item():.2f} at position {X_train[0].argmax().item()}")

### Mini-batches with `DataLoader` 📦

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=256)

xb, yb = next(iter(train_loader))
print("one batch:", xb.shape, yb.shape)
print("batches per epoch:", len(train_loader))

## The attention layer ⚙️

We implement **additive attention pooling**: a tiny one-hidden-layer network scores
each hidden state. This uses the nonlinear scoring idea of Bahdanau-style attention,
but has no separate query. The original encoder–decoder mechanism also conditions
each score on the decoder state.

$$e_t \;=\; v^\top \tanh\!\big(W h_t + b\big)$$

$$\alpha \;=\; \text{softmax}(e), \qquad c \;=\; \sum_t \alpha_t h_t$$

- $W$ projects the hidden state into a small **attention space** (`attn_units`),
- $\tanh$ adds a non-linearity, so the score can be a non-trivial function of $h_t$,
- $v$ collapses the result into **one number per time step**.

Two implementation details worth noticing:

- `masked_fill(~mask, -inf)` — positions we want to **forbid** get a score of $-\infty$,
  which softmax turns into a weight of exactly **0**. Each row must keep at least one
  position; softmax over an entirely masked row would produce NaNs.
- `forward()` returns **both** the context **and** the weights. In PyTorch we simply return
  whatever we want from `forward()` — no tricks are needed to look inside the model. 🎉

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_size, attn_units=32):
        super().__init__()
        self.W = nn.Linear(hidden_size, attn_units)      # hidden -> attention space
        self.v = nn.Linear(attn_units, 1, bias=False)    # attention space -> one score

    def forward(self, H, mask=None):
        # H: (batch, time, hidden), mask: (batch, time) with True = real token
        scores = self.v(torch.tanh(self.W(H)))           # (batch, time, 1)
        scores = scores.squeeze(-1)                      # (batch, time)

        if mask is not None:
            if mask.dtype != torch.bool or mask.shape != scores.shape:
                raise ValueError("mask must be boolean with shape (batch, time)")
            if not mask.any(dim=1).all():
                raise ValueError("Each sequence must contain at least one unmasked token")
            scores = scores.masked_fill(~mask, float("-inf"))   # forbidden -> weight 0

        weights = torch.softmax(scores, dim=1)           # (batch, time), sums to 1
        context = torch.bmm(weights.unsqueeze(1), H)     # (batch, 1, hidden)
        context = context.squeeze(1)                     # (batch, hidden)
        return context, weights

In [ ]:
# Quick shape check on random data - always do this before training!
att_test = AdditiveAttention(hidden_size=64, attn_units=32)
ctx, wts = att_test(torch.randn(8, 15, 64))

print("context:", ctx.shape, "| weights:", wts.shape)
print("each row of weights sums to 1:", torch.allclose(wts.sum(dim=1), torch.ones(8)))

## The model 🏗️

$$\text{sequence} \rightarrow \text{LSTM (all 64-dim hidden states)} \rightarrow \text{attention} \rightarrow \text{context} \rightarrow \text{Linear} \rightarrow \hat{y}$$

- `nn.LSTM(..., batch_first=True)` makes the tensors `(batch, time, features)` — the same
  ordering we used above. Without it PyTorch expects `(time, batch, features)`. ⚠️
- The LSTM returns **all** hidden states `H` (that is what we need!) plus the final states.
- The output layer has **no activation** — this is regression, we predict a raw number.

In [ ]:
class FindMaxNet(nn.Module):
    def __init__(self, hidden_size=64, attn_units=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.attention = AdditiveAttention(hidden_size, attn_units)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        H, (h_n, c_n) = self.lstm(x)            # H: (batch, time, hidden)
        context, weights = self.attention(H)    # context: (batch, hidden)
        out = self.head(context)                # (batch, 1)
        return out, weights                     # we return the weights for visualisation

In [ ]:
model_max = FindMaxNet().to(device)
print(model_max)

In [ ]:
torchinfo.summary(model_max, input_size=(BATCH_SIZE, SEQ_LEN, 1), device=device, mode="eval")

## Training 🚀
Loss: **MSE** (mean squared error), a standard choice for regression.
We restore the checkpoint with the lowest validation loss before evaluating the test set.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_max.parameters(), lr=2e-3)

EPOCHS = 80
history = {"train_loss": [], "val_loss": []}

best_val, best_state = float("inf"), None
start = time.perf_counter()
for epoch in range(EPOCHS):
    # ---- training -------------------------------------------------
    model_max.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        preds, _ = model_max(xb)          # forward
        loss = criterion(preds, yb)       # loss

        optimizer.zero_grad()             # reset gradients
        loss.backward()                   # backward
        optimizer.step()                  # update

        running += loss.item() * xb.size(0)
    train_loss = running / len(train_loader.dataset)

    # ---- validation (no gradients!) -------------------------------
    model_max.eval()
    running = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds, _ = model_max(xb)
            running += criterion(preds, yb).item() * xb.size(0)
    val_loss = running / len(val_loader.dataset)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model_max.state_dict().items()}

    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1:3d} | train MSE {train_loss:.4f} | val MSE {val_loss:.4f}")

if best_state is None:
    raise RuntimeError("Training did not produce a finite validation loss")
model_max.load_state_dict(best_state)
print(f"\nTraining finished in {time.perf_counter() - start:.1f} s")
print(f"Restored checkpoint with validation MSE {best_val:.4f}")

In [ ]:
plt.figure(figsize=(6, 3.5))
for key, values in history.items():
    plt.plot(values, label=key)

plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.yscale("log")           # log scale - the loss drops over orders of magnitude
plt.legend()
plt.tight_layout()
plt.show()

## Evaluation 📊

First the numbers, then the picture.

In [ ]:
model_max.eval()
with torch.no_grad():
    test_preds, test_weights = model_max(X_test.to(device))

test_preds   = test_preds.cpu()
test_weights = test_weights.cpu()

mse = criterion(test_preds, y_test).item()
mae = (test_preds - y_test).abs().mean().item()
print(f"Test MSE: {mse:.4f}")
print(f"Test MAE: {mae:.4f}   (average error in the original units)")

In [ ]:
plt.figure(figsize=(4.2, 4))
sns.scatterplot(x=y_test.squeeze(-1).numpy(), y=test_preds.squeeze(-1).numpy(), s=12, alpha=.5)
plt.plot([0, 10], [0, 10], "r--", label="perfect prediction")
plt.xlabel("true maximum"); plt.ylabel("predicted maximum")
plt.legend(); plt.tight_layout(); plt.show()

### Does the attention really look at the maximum? 🔍

This is the interesting part. For every test sequence we compare:

- `argmax` of the **sequence** — where the true maximum is,
- `argmax` of the **attention weights** — the most heavily weighted hidden state.

In [ ]:
true_pos = X_test.squeeze(-1).argmax(dim=1)
att_pos  = test_weights.argmax(dim=1)

exact  = (att_pos == true_pos).float().mean().item()
within = ((att_pos - true_pos).abs() <= 1).float().mean().item()

print(f"Attention peak exactly on the maximum : {exact:.1%}")
print(f"Attention peak within +/- 1 position  : {within:.1%}")

The loss supervises the **predicted maximum**, not the attention weights. A peak near the
maximum is one possible learned strategy; a later state may already encode that maximum.
Interpret the alignment scores alongside the regression error.

Let's plot 6 different test sequences. Blue = the values, red = the attention weights
(rescaled so that both fit in one plot).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))

for i, ax in enumerate(axes.ravel()):
    seq = X_test[i].squeeze(-1).numpy()
    att = test_weights[i].numpy()

    ax.plot(seq, "b-o", ms=4, label="sequence values")
    ax.plot(att * seq.max() / max(att.max(), 1e-9), "r--s", ms=4, label="attention (rescaled)")
    ax.axvline(seq.argmax(), color="green", ls=":", lw=2)

    ax.set_title(f"max={y_test[i].item():.2f} | true pos={seq.argmax()} | attention pos={att.argmax()}",
                 fontsize=9)
    ax.set_xlabel("position in the sequence")
    if i == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

> 💡 The maximum can occur at any position. Low error on independently generated test
> sequences is evidence of generalization within this data distribution. The attention
> plots help inspect the model, but do not prove its strategy or guarantee performance
> on different sequence lengths or number ranges.

# 4. Task 2: Sentiment analysis 💬

Now something real. We classify **short customer reviews** (restaurants, products, films) as
**positive (1)** or **negative (0)**. Each review is a single sentence, which is exactly the
kind of data where every word counts.

We will train **two models** and compare them:

| Model | Sequence summary |
|---|---|
| **Baseline** | the hidden state after the **last real token** |
| **Attention** | a **weighted sum** of all hidden states |

Both models use the same split, embedding and LSTM sizes, optimizer settings and
padding handling. Shared layers start from the same initialization; we also reset
the training seed and batch-order generator for each run. A single run is still a
noisy comparison, so repeat across seeds before drawing strong conclusions.

We will also visualize which **hidden states** receive high pooling weights. 🔍

In [ ]:
url = "https://raw.githubusercontent.com/rasvob/VSB-FEI-Deep-Learning-Exercises/main/datasets/yelp_labelled.txt"
path = Path("yelp_labelled.txt")
if not path.exists():
    urllib.request.urlretrieve(url, path)

# The course mirror combines review domains despite its yelp_labelled.txt filename.
# Each nonempty line must be: <review text> \t <label 0/1>.
rows = []
with path.open(encoding="utf-8") as source:
    for line_number, line in enumerate(source, start=1):
        line = line.strip()
        if not line:
            continue
        parts = line.rsplit("\t", 1)
        if len(parts) != 2 or parts[1].strip() not in ("0", "1"):
            raise ValueError(f"Malformed review at line {line_number}")
        text, label = parts
        rows.append({"Text": text.strip(), "Label": int(label)})

df = pd.DataFrame(rows, columns=["Text", "Label"])
if df.empty:
    raise ValueError("The review file contains no labelled examples")
print("rows before deduplication:", len(df))
df.head()

In [ ]:
print(df.Label.value_counts())
print("\nA positive one:", df[df.Label == 1].Text.iloc[0])
print("A negative one:", df[df.Label == 0].Text.iloc[0])

> 💡 The source dataset is balanced, so accuracy is a useful first metric here. We remove
> duplicate tokenized reviews before splitting and print the resulting class counts.
> `stratify` preserves these class proportions approximately in each subset.

In [ ]:
df["Text_length"] = df["Text"].str.split().str.len()

plt.hist(df.Text.str.split().str.len(), bins=30)
plt.xlabel("Text Length (words)")
plt.ylabel("Number of Reviews")
plt.title("Histogram of Review Lengths")
plt.show()

## From text to tensors 🔤

A network cannot read strings. The pipeline is always the same:

1. **Tokenisation** — lowercase and retain sequences of letters, digits and apostrophes,
2. **Vocabulary** — map each word to an integer id; keep only the most frequent `MAX_TOKENS`,
3. **Encoding + padding** — turn each review into a fixed-length vector of ids.

Two special ids are reserved:

| id | token | meaning |
|---|---|---|
| `0` | `<pad>` | filler, so that all reviews have the same length |
| `1` | `<unk>` | any word that is not in the vocabulary |

> ⚠️ The vocabulary is built from the **training set only**. Building it from all the data
> would leak information from the test set into training. Before splitting, we remove
> repeated token sequences and ambiguous sequences with conflicting labels, so that
> copies of a review cannot appear in both training and evaluation sets.

In [ ]:
def tokenize(text):
    '''Lowercase the text and keep sequences of letters/digits/apostrophes.'''
    return re.findall(r"[a-z0-9']+", text.lower())


print(tokenize("The food was GREAT, but the service wasn't..."))

In [ ]:
# Deduplicate using the same normalization as the model, before splitting.
df = df.copy()
df["token_key"] = df.Text.map(lambda text: " ".join(tokenize(text)))
before = len(df)
df = df.loc[df.token_key.ne("")].copy()
label_counts = df.groupby("token_key")["Label"].transform("nunique")
df = df.loc[label_counts.eq(1)].drop_duplicates("token_key").reset_index(drop=True)
print(f"Removed {before - len(df)} empty, duplicate or conflicting-label reviews")
print("Class counts after cleaning:\n", df.Label.value_counts())

X_temp, X_test, y_temp, y_test = train_test_split(
    df.Text.values, df.Label.values,
    test_size=0.15, random_state=SEED, stratify=df.Label.values,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15 / 0.85, random_state=SEED, stratify=y_temp,
)

# Approximately 70% / 15% / 15%; integer split sizes require rounding.
print(f"train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")

In [ ]:
MAX_TOKENS = 3000     # vocabulary size
MAX_LEN    = 32  
PAD_ID, UNK_ID = 0, 1

counter = Counter(tok for text in X_train for tok in tokenize(text))

itos = ["<pad>", "<unk>"] + [w for w, _ in counter.most_common(MAX_TOKENS - 2)]  # id -> word
stoi = {w: i for i, w in enumerate(itos)}                                        # word -> id

print("vocabulary size:", len(itos))
print("most frequent words:", itos[2:12])

In [ ]:
def encode(texts):
    '''Text -> tensor of ids of shape (n_texts, MAX_LEN), padded with PAD_ID.'''
    if MAX_LEN < 1:
        raise ValueError("MAX_LEN must be positive")
    out = torch.full((len(texts), MAX_LEN), PAD_ID, dtype=torch.long)
    for i, text in enumerate(texts):
        ids = [stoi.get(tok, UNK_ID) for tok in tokenize(text)][:MAX_LEN]   # truncate
        ids = ids or [UNK_ID]  # empty/punctuation-only input still has one valid token
        out[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)             # the rest stays <pad>
    return out


X_tr, X_va, X_te = encode(X_train), encode(X_val), encode(X_test)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print("X_tr:", X_tr.shape, "| y_tr:", y_train_tensor.shape)

In [ ]:
# What one review looks like after encoding
print("original :", X_train[0])
print("ids      :", X_tr[0].tolist())
print("decoded  :", " ".join(itos[i] for i in X_tr[0].tolist()))
avg_len = (X_tr != PAD_ID).float().sum(1).mean().item()
print(f"\nreal tokens per review (average): {avg_len:.1f} out of {MAX_LEN}")

> 🔎 Compare the decoded example with the average token count. Padding varies by review;
> `<pad>` positions must not affect either the recurrent summary or attention pooling.

In [ ]:
train_loader_txt = DataLoader(
    TensorDataset(X_tr, y_train_tensor), batch_size=32, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader_txt = DataLoader(TensorDataset(X_va, y_val_tensor), batch_size=128)

## Baseline: LSTM without attention

$$\text{ids} \rightarrow \text{Embedding} \rightarrow \text{packed LSTM} \rightarrow h_{\text{last real}} \rightarrow \text{Dropout} \rightarrow \text{Linear} \rightarrow \text{logit}$$

- **`nn.Embedding(vocab, dim, padding_idx=0)`** learns a vector per word for this task.
  `padding_idx=0` keeps the `<pad>` embedding at zero, but a zero input would still update
  the LSTM's recurrent state if we processed the padded positions.
- **`pack_padded_sequence`** tells the LSTM each review's real length. With packed inputs,
  `h_n[-1]` is the state after the **last real token**, not after trailing padding.
- The output is a single **logit**; we use **`BCEWithLogitsLoss`**.

See the [PyTorch packing documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html).

In [ ]:
EMB_DIM, HID_DIM = 64, 64

class BaselineLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=EMB_DIM, hidden_size=HID_DIM):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(emb_dim, hidden_size, batch_first=True)
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lengths = (x != PAD_ID).sum(dim=1).cpu()
        packed = pack_padded_sequence(
            self.emb(x), lengths, batch_first=True, enforce_sorted=False,
        )
        _, (h_n, _) = self.lstm(packed)
        summary = h_n[-1]                 # (batch, hidden), after the last REAL token
        return self.head(self.drop(summary))

In [ ]:
torch.manual_seed(SEED)
model_base = BaselineLSTM(len(itos)).to(device)
torchinfo.summary(model_base, input_data=X_tr[:32].to(device), mode="eval")

### A reusable training function 🔁

In [ ]:
def train_classifier(model, epochs=15, lr=1e-3, returns_weights=False):
    if epochs < 1:
        raise ValueError("epochs must be positive")
    torch.manual_seed(SEED)
    train_loader_txt.generator.manual_seed(SEED)
    start = time.perf_counter()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val, best_state = float("inf"), None

    for epoch in range(epochs):
        # ---- training ---------------------------------------------
        model.train()
        running = 0.0
        for xb, yb in train_loader_txt:
            xb, yb = xb.to(device), yb.to(device)

            out = model(xb)
            logits = out[0] if returns_weights else out
            loss = criterion(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
        train_loss = running / len(train_loader_txt.dataset)

        # ---- validation -------------------------------------------
        model.eval()
        running, correct = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader_txt:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                logits = out[0] if returns_weights else out
                running += criterion(logits, yb).item() * xb.size(0)
                preds = (torch.sigmoid(logits) >= 0.5).float()
                correct += (preds == yb).sum().item()
        val_loss = running / len(val_loader_txt.dataset)
        val_acc  = correct / len(val_loader_txt.dataset)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_loss < best_val:                       # keep the best checkpoint
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"epoch {epoch+1:2d} | train {train_loss:.4f} | val {val_loss:.4f} | val acc {val_acc:.3f}")

    if best_state is None:
        raise RuntimeError("Training did not produce a finite validation loss")
    model.load_state_dict(best_state)                 # restore the best weights
    print(f"\nBest validation loss: {best_val:.4f} - these weights were restored.")
    print(f"Training time: {time.perf_counter() - start:.1f} s")
    return history

In [ ]:
def evaluate(model, X, y, returns_weights=False):
    model.eval()
    with torch.no_grad():
        out = model(X.to(device))
        logits = out[0] if returns_weights else out
        preds = (torch.sigmoid(logits) >= 0.5).float().cpu()
    return (preds == y).float().mean().item()

In [ ]:
def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(history["train_loss"], label="train_loss")
    ax[0].plot(history["val_loss"], label="val_loss")
    ax[0].set_xlabel("epoch")
    ax[0].set_ylabel("BCE loss")
    ax[0].legend()
    ax[1].plot(history["val_acc"], color="tab:green", label="val_acc")
    ax[1].set_xlabel("epoch")
    ax[1].set_ylabel("accuracy")
    ax[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
hist_base = train_classifier(model_base, epochs=15)

In [ ]:
plot_history(hist_base, "Baseline LSTM (last hidden state)")

In [ ]:
acc_base = evaluate(model_base, X_te, y_test_tensor)
print(f"Baseline test accuracy: {acc_base:.3f}")

> 🔎 If training loss keeps falling while validation loss rises, that is a sign of
> **overfitting**. This is a risk with a small dataset, not an inevitable outcome of every
> run. We select checkpoints by validation loss and reserve the test set for final reporting.

## The same model + attention

One difference: instead of `h_n[-1]` we pool **all** hidden states with our
`AdditiveAttention` — **and we pass the mask**, so that `<pad>` positions get weight 0.
Both models pack the LSTM inputs. Here we unpack its outputs to the original padded
width so that every attention weight still aligns with its input token.

`mask = (x != PAD_ID)` is a boolean tensor of shape `(batch, time)`: `True` = real word.

In [ ]:
class AttentionLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=EMB_DIM, hidden_size=HID_DIM, attn_units=32):
        super().__init__()
        # Initialize shared layers in the same order as BaselineLSTM.
        self.emb       = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.lstm      = nn.LSTM(emb_dim, hidden_size, batch_first=True)
        self.drop      = nn.Dropout(0.3)
        self.head      = nn.Linear(hidden_size, 1)
        self.attention = AdditiveAttention(hidden_size, attn_units)

    def encode_hidden(self, x):
        mask = x != PAD_ID
        lengths = mask.sum(dim=1).cpu()
        packed = pack_padded_sequence(
            self.emb(x), lengths, batch_first=True, enforce_sorted=False,
        )
        packed_H, _ = self.lstm(packed)
        H, _ = pad_packed_sequence(packed_H, batch_first=True, total_length=x.size(1))
        return H, mask                         # padded output states are zero

    def forward(self, x):
        H, mask = self.encode_hidden(x)
        context, weights = self.attention(H, mask)
        return self.head(self.drop(context)), weights

In [ ]:
torch.manual_seed(SEED)
model_attn = AttentionLSTM(len(itos)).to(device)
torchinfo.summary(model_attn, input_data=X_tr[:32].to(device), mode="eval")

attention_params = sum(p.numel() for p in model_attn.attention.parameters())
total_params = sum(p.numel() for p in model_attn.parameters())
print(f"Attention parameters: {attention_params:,} ({attention_params / total_params:.2%} of model)")

> 💡 With 64 hidden features and 32 attention units, this pooling layer adds **2,112**
> parameters. Its fraction of the full model depends on vocabulary size; use the printed
> count rather than assuming it is always below 1%. This small pooling layer is different
> from the quadratic attention matrix used by full self-attention later in the notebook.

In [ ]:
hist_attn = train_classifier(model_attn, epochs=15, returns_weights=True)

In [ ]:
plot_history(hist_attn, "LSTM + additive attention")

In [ ]:
acc_attn = evaluate(model_attn, X_te, y_test_tensor, returns_weights=True)

results = pd.DataFrame({
    "model": ["Baseline (last real hidden state)", "Attention (weighted sum)"],
    "params": [sum(p.numel() for p in model_base.parameters()),
               sum(p.numel() for p in model_attn.parameters())],
    "test accuracy": [acc_base, acc_attn],
})
results

## Which hidden states received high attention? 🔍

The weights $\alpha$ sum to one over the real token positions. We display the token at each
position as a label, but its LSTM state also includes earlier context. These plots show
how the model pools states, not a causal importance score for each individual word.

In [ ]:
def show_attention(model, X_source, y_source, n=4, show_padding=False):
    '''Plot pooling weights; optionally include padding for the mask ablation.'''
    n = min(n, len(X_source))
    if n < 1:
        raise ValueError("At least one review is needed for a plot")
    model.eval()
    with torch.no_grad():
        logits, weights = model(X_source[:n].to(device))
    probs = torch.sigmoid(logits).cpu().squeeze(-1)
    weights = weights.cpu()

    fig, axes = plt.subplots(n, 1, figsize=(13, 1.7 * n))
    for i, ax in enumerate(np.atleast_1d(axes)):
        ids = X_source[i].tolist()
        n_tok = sum(1 for t in ids if t != PAD_ID)
        width = len(ids) if show_padding else n_tok
        toks = [itos[t] for t in ids[:width]]
        w = weights[i, :width].numpy()
        pad_mass = weights[i, n_tok:].sum().item()
        vmax = max(w.max(), 1e-6)

        ax.imshow(w[None, :], cmap="Reds", aspect="auto", vmin=0, vmax=vmax)

        # --- the numbers inside the cells ---
        fontsize = 9 if width <= 16 else (7 if width <= 24 else 6)
        for k, value in enumerate(w):
            ax.text(k, 0, f"{value:.2f}", ha="center", va="center", fontsize=fontsize,
                    color="white" if value > 0.55 * vmax else "black")

        ax.set_xticks(range(width))
        ax.set_xticklabels(toks, rotation=45, ha="right", fontsize=8)
        ax.set_yticks([])
        ax.grid(False)
        truth = "positive" if y_source[i].item() == 1 else "negative"
        ax.set_title(f"true: {truth} | P(positive)={probs[i]:.2f} | pad mass={pad_mass:.2f}",
                     fontsize=9)

    plt.tight_layout()
    plt.show()

show_attention(model_attn, X_te, y_test_tensor, n=4)

In [ ]:
# Plain-text view: the three most heavily weighted token positions per review
model_attn.eval()
with torch.no_grad():
    logits, weights = model_attn(X_te[:8].to(device))

for i in range(8):
    ids = X_te[i].tolist()
    n_tok = sum(1 for t in ids if t != PAD_ID)
    toks = [itos[t] for t in ids[:n_tok]]
    w = weights[i, :n_tok].cpu().numpy()
    top3 = set(np.argsort(-w)[:3].tolist())
    marked = " ".join(f"[{t}]" if k in top3 else t for k, t in enumerate(toks))
    print(f"P(pos)={torch.sigmoid(logits[i]).item():.2f} | true={y_test[i]} | {marked}\n")

> 💡 Brackets mark the top three token positions by pooling weight. Some may be sentiment
> words; others may summarize earlier context. The weights are learned from review labels,
> with no supervision about where the attention peak should be.

## ⚠️ Why the mask really matters

Remember the question about `<pad>`? Time to answer it experimentally.

We take the **trained** attention layer and run it **twice** on the same hidden states —
once with the mask, once without.

In [ ]:
sample = X_te[:200].to(device)

model_attn.eval()
with torch.no_grad():
    H, real_mask = model_attn.encode_hidden(sample)
    _, w_masked = model_attn.attention(H, mask=real_mask)
    _, w_unmasked = model_attn.attention(H, mask=None)

pad_mask = ~real_mask
pad_mass_masked = (w_masked * pad_mask).sum(dim=1)
pad_mass_unmasked = (w_unmasked * pad_mask).sum(dim=1)

print(f"attention mass on <pad> WITH mask   : {pad_mass_masked.mean():.4f}")
print(f"attention mass on <pad> WITHOUT mask: {pad_mass_unmasked.mean():.4f}")

Without the mask, padded positions can receive attention even though their unpacked hidden
states are zero. The scorer still assigns them a finite score, and they participate in
the softmax denominator. Their total weight depends on the scores and padding length;
use the measured value rather than assuming it will always be large.

> 💡 Exclude padded positions from attention over keys and from any final pooling over
> sequence positions. A key-padding mask alone does not remove outputs for padded queries.

# 5. Modern attention ⚡


Modern architectures generalise this. **Every position asks its own question** about every
other position. That is **self-attention**, and it is the building block of transformers
(next lecture 👀).

### Queries, Keys and Values

The vocabulary comes from information retrieval:

| Symbol | Name | Intuition |
|---|---|---|
| $Q$ | **Query** | what am I looking for? |
| $K$ | **Key** | what do I offer? |
| $V$ | **Value** | what do I hand over when chosen? |

Scores are now **dot products** between queries and keys. The scoring step has no
MLP, although learned linear projections normally produce $Q$, $K$ and $V$:

$$\text{Attention}(Q,K,V) \;=\; \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

### Why divide by $\sqrt{d_k}$? 🤔

For independent, zero-mean, unit-variance query/key components, the dot product has
variance $d_k$.
For large $d_k$ the scores become huge, softmax saturates into a near one-hot vector, and the
gradients vanish. Dividing by $\sqrt{d_k}$ keeps the scores in a healthy range — hence the name
**scaled** dot-product attention.

Note the shape of the result: for $T$ positions the softmax argument is a $T \times T$ matrix —
**every position can attend to every position**, unless a mask forbids it.

Reference: [Attention Is All You Need](https://arxiv.org/abs/1706.03762).

In [ ]:
torch.manual_seed(SEED)

B, T, D = 1, 6, 8          # batch, time steps, dimension
Q = torch.randn(B, T, D)
K = torch.randn(B, T, D)
V = torch.randn(B, T, D)

scores = (Q @ K.transpose(-2, -1)) / (D ** 0.5)   # (B, T, T)
A = torch.softmax(scores, dim=-1)                 # every ROW sums to 1
out_manual = A @ V                                # (B, T, D)

print("attention matrix:", A.shape, "| output:", out_manual.shape)
print("row sums:", A.sum(dim=-1))

PyTorch ships the same operation as a highly optimised built-in function
(**`F.scaled_dot_product_attention`**, which can dispatch to FlashAttention on a GPU).
Let's verify agreement within floating-point tolerance. ✅

In [ ]:
out_builtin = F.scaled_dot_product_attention(Q, K, V)

print("results match within tolerance:", torch.allclose(out_manual, out_builtin, atol=1e-6))
print("max difference   :", (out_manual - out_builtin).abs().max().item())

### Self-attention on a real sentence

**Self-attention** means that queries, keys and values come from the **same sequence**.
Usually $Q=XW_Q$, $K=XW_K$ and $V=XW_V$, so they need not be equal.

For a simple demonstration below, we use identity projections: $Q=K=V=X$, taking the
**trained embeddings** from our sentiment model. Bright cells indicate high normalized
dot-product scores. Vector norms and self-matches affect these scores, so this is not
automatically a map of semantic relationships. This demonstration also adds no position
encoding; the LSTM used in the classifier supplies sequence order through recurrence.

In [ ]:
review_ids = X_te[3]
n_tok = int((review_ids != PAD_ID).sum())
tok_labels = [itos[i] for i in review_ids[:n_tok].tolist()]

with torch.no_grad():
    X_emb = model_attn.emb(review_ids[:n_tok].to(device)).unsqueeze(0)   # (1, n_tok, EMB_DIM)
    self_scores = (X_emb @ X_emb.transpose(-2, -1)) / (EMB_DIM ** 0.5)
    A_self = torch.softmax(self_scores, dim=-1)[0].cpu()

plt.figure(figsize=(min(1 + 0.45 * n_tok, 12), min(1 + 0.45 * n_tok, 12)))
sns.heatmap(A_self, xticklabels=tok_labels, yticklabels=tok_labels, cmap="viridis", square=True)
plt.xlabel("attends to (Key)"); plt.ylabel("asking (Query)")
plt.title("Self-attention over the learned embeddings")
plt.tight_layout(); plt.show()

### Masks, take two 🎭

Two masks appear everywhere in modern models:

- **padding mask** — the one we already used: ignore `<pad>` positions,
- **causal (look-ahead) mask** — a position may only attend to itself and to the **past**.
  This is what makes a language model unable to cheat by peeking at the next word.

Both are implemented the same way: set the forbidden scores to $-\infty$ before the softmax.

In [ ]:
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))     # lower triangle = allowed

scores = (Q @ K.transpose(-2, -1)) / (D ** 0.5)             # the same scores as above
masked_scores = scores.masked_fill(~causal, float("-inf"))  # the future gets -inf
A_causal = torch.softmax(masked_scores, dim=-1)

print(A_causal[0].round(decimals=2))
print("\nsame as the built-in version:",
      torch.allclose(A_causal @ V,
                     F.scaled_dot_product_attention(Q, K, V, is_causal=True), atol=1e-6))

Read the printed matrix: the first row can only use position 0, the second row positions 0–1,
and so on. The upper triangle is exactly zero. 🔺

### Multi-head attention 🧠🧠🧠

Multi-head attention computes several attention distributions in parallel using different
learned query/key/value projections. It concatenates their outputs and applies a learned
output projection. Heads can learn different patterns; a particular specialization is
not guaranteed.

PyTorch provides **`nn.MultiheadAttention`**. The instance below is **randomly initialized**:
we inspect its interface and tensor shapes, not learned linguistic relationships.

In [ ]:
mha = nn.MultiheadAttention(embed_dim=EMB_DIM, num_heads=4, batch_first=True).to(device)
mha.eval()

batch = X_te[:2].to(device)
pad_mask = (batch == PAD_ID)          # NOTE: here True means "ignore this position"!

with torch.no_grad():
    x = model_attn.emb(batch)
    out, attn_w = mha(x, x, x, key_padding_mask=pad_mask,
                      need_weights=True, average_attn_weights=True)

print("output       :", out.shape)          # (batch, time, embed_dim)
print("weights (avg):", attn_w.shape)       # (batch, time, time)

with torch.no_grad():
    _, attn_per_head = mha(x, x, x, key_padding_mask=pad_mask,
                           need_weights=True, average_attn_weights=False)
print("weights per head:", attn_per_head.shape)   # (batch, heads, time, time)

> ⚠️ Boolean mask conventions differ:
> - our `AdditiveAttention` and `F.scaled_dot_product_attention`: `True` means **keep**,
> - `nn.MultiheadAttention`'s `key_padding_mask` and boolean `attn_mask`: `True` means **ignore**.
>
> The key-padding mask removes padded **keys**, not padded **query rows**. Exclude padded
> output positions separately when pooling a sequence for classification.
> See the [PyTorch mask documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html).

# 6. Activities ✏️

Use **validation** results to choose architectures and hyperparameters. After choosing
your configuration, report its test accuracy once; repeated tuning on test scores
would make the test set part of model selection. The empty cells are for your solutions.

## Activity 1 — Dot-product instead of additive scoring 🔁

Our `AdditiveAttention` scores a position with a small network, $e_t = v^\top \tanh(W h_t + b)$.
The cheaper alternative is a **plain dot product with a single learnable vector**:
$e_t = h_t \cdot w$ — exactly what we did by hand in Part 2.

**What to do**

1. Write a new module `DotProductAttention(nn.Module)`. In `__init__` create a single
   learnable vector, e.g. `self.w = nn.Parameter(torch.randn(hidden_size) * 0.1)`
   (or equivalently a `nn.Linear(hidden_size, 1, bias=False)`).
2. In `forward(H, mask=None)` compute the scores as `H @ self.w`, apply the same
   `masked_fill` + `softmax` + weighted sum, and return `(context, weights)` — keep the
   interface identical, including the guard against fully masked rows.
3. Build a second `AttentionLSTM` that uses it (the easiest way is to instantiate the model
   and then overwrite `model.attention = DotProductAttention(HID_DIM).to(device)`),
   train it with `train_classifier(..., returns_weights=True)` and evaluate on validation data.
4. Compare with the additive version: **validation accuracy**, **number of parameters**
   (`sum(p.numel() for p in m.parameters())`) and **training time**.

## Activity 2 — Bidirectional LSTM 🔄

A normal LSTM reads left to right, so the hidden state at position $t$ knows nothing about the
words that follow. `nn.LSTM(..., bidirectional=True)` runs a second LSTM backwards and
**concatenates** both hidden states.

**What to do**

1. Create a variant of `AttentionLSTM` with `bidirectional=True` in the LSTM. Keep
   the packing/unpacking in `encode_hidden`: otherwise the backward LSTM would read
   trailing padding before it reaches the real words.
2. ⚠️ The hidden states are now **twice as wide**, so the attention layer and the output layer
   must be built with `hidden_size * 2`. Fix the dimensions — a shape error here is the point
   of the exercise.
3. Train it and compare validation accuracy with the one-directional model.
4. Print the parameter count of both models.

## Activity 3 — Attention without a mask 🐛

We measured the padding weight after removing a trained layer's mask. Now train a model
whose attention **never used a mask**, and check whether it learns to suppress padding.

**What to do**

1. Copy `AttentionLSTM` and pass `mask=None` to its attention layer. Keep the packed LSTM
   encoder unchanged so that only the pooling mask differs.
2. Train it with the same settings and compare validation accuracy.
3. Call `show_attention(..., show_padding=True)` on short reviews to make padding visible.
4. Measure the mean attention mass on `<pad>` positions as above. Report what happens;
   neither a large padding weight nor worse accuracy is guaranteed for every run.

## Activity 4 — Multi-head attention 🧠🧠

Replace our single-summary attention with multi-head self-attention.

**What to do**

1. Build a model: `Embedding → packed LSTM (all states) → MultiheadAttention
   → mean over real tokens → Linear`.
   - unpack to the input width before attention,
   - construct `nn.MultiheadAttention(embed_dim=HID_DIM, num_heads=h, batch_first=True)`;
     the embedding dimension must be divisible by the number of heads,
   - call `mha(H, H, H, key_padding_mask=(x == PAD_ID))`,
   - exclude padded query positions when averaging: multiply by
     `(x != PAD_ID).unsqueeze(-1)` and divide by the number of real tokens,
   - return logits and train with `train_classifier(model)`.
2. Train with `num_heads = 1`, `2` and `4`; compare **validation** accuracy.
3. In evaluation mode, extract per-head matrices with `average_attn_weights=False` for
   one review. Plot the real-token rows and columns as separate heatmaps.